# Tutorial 13-3: Squeezing LLMs – "4-bit Quantization with QLoRA"

**Course:** CSEN 342: Deep Learning  
**Topic:** Extreme Quantization, QLoRA, PEFT, and Large Language Models

## Objective
Training Large Language Models (LLMs) usually requires massive GPU clusters. However, recent advances in **Resource-Aware Deep Learning** have democratized this.

As mentioned in the lecture (Slide 99), **QLoRA (Quantized Low-Rank Adaptation)** combines two techniques:
1.  **4-bit Quantization:** Squeezing the base model weights into 4-bit precision (specifically **NF4**, NormalFloat-4) to save memory.
2.  **LoRA:** Freezing the base model and training only tiny "adapter" matrices.

In this tutorial, we will take a **1.3 Billion Parameter** model (`facebook/opt-1.3b`), compress it to 4-bit, and prepare it for fine-tuning on a single GPU.

---

## Part 1: Setup and Dependencies

We need the `bitsandbytes`, `peft`, and `accelerate` libraries from Hugging Face. These perform the heavy lifting of quantization and adapter management.

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import os

# Check for GPU
if not torch.cuda.is_available():
    raise RuntimeError("This tutorial requires a GPU! QLoRA relies on CUDA kernels.")

device = "cuda"
model_id = "facebook/opt-1.3b" # A medium-sized LLM

print(f"Using GPU: {torch.cuda.get_device_name(0)}")

---

## Part 2: 4-bit Loading (The Squeeze)

We define a `BitsAndBytesConfig` to tell the library how to load the model.
* `load_in_4bit=True`: The main switch.
* `bnb_4bit_quant_type="nf4"`: Use **NormalFloat 4**, a data type optimized for the bell-curve distribution of neural network weights (better than standard Int4).
* `bnb_4bit_compute_dtype=torch.float16`: We store weights in 4-bit, but dequantize them to FP16 for computation.

In [ ]:
# 1. Define Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True, # Quantize the quantization constants (extra savings)
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 2. Load Model
print(f"Loading {model_id} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

# 3. Verify Memory Footprint
mem_params = sum([param.nelement() * param.element_size() for param in model.parameters()])
mem_buffers = sum([buf.nelement() * buf.element_size() for buf in model.buffers()])
model_mem_mb = (mem_params + mem_buffers) / 1024**2

print(f"Model Memory Footprint: {model_mem_mb:.2f} MB")
print("Note: In FP16, this would be ~2600 MB. We achieved ~3-4x compression!")

---

## Part 3: Adding LoRA Adapters

The model is now loaded in 4-bit. However, **we cannot train 4-bit weights directly** (gradients don't flow well through quantized discrete values).

**Solution:** We freeze the 4-bit base model and attach **LoRA adapters**. These are small, full-precision matrices that learn the *changes* to the weights.

$$ W_{new} = W_{frozen} + (A \times B) $$

In [ ]:
# 1. Prepare for k-bit training
# This helper freezes weights and casts layer norms to float32 for stability
model = prepare_model_for_kbit_training(model)

# 2. Define LoRA Config
peft_config = LoraConfig(
    r=16,           # Rank of the adapter (lower = fewer params)
    lora_alpha=32,  # Scaling factor
    target_modules=["k_proj", "v_proj", "q_proj", "out_proj"], # Attach to attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. Inject Adapters
model = get_peft_model(model, peft_config)

# 4. Print Trainable Parameters
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.4f}"
    )

print_trainable_parameters(model)

### Discussion
Look at the output above. You should see that we are training less than **1%** of the parameters. This means we only need to store gradients/optimizer states for that tiny fraction, massively reducing VRAM usage during training.

---

## Part 4: Inference Check

Before training, let's verify the model still works. Even though it's quantized, it should generate coherent text.

In [ ]:
prompt = "The future of artificial intelligence is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=30, 
        do_sample=True, 
        temperature=0.7
    )

print("--- Generation ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Part 5: The Training Loop (Proof of Concept)

We will run a single forward/backward pass to prove that gradients flow only to the adapters.

In [ ]:
# Dummy Data
dummy_text = "This tutorial demonstrates QLoRA fine-tuning."
tokens = tokenizer(dummy_text, return_tensors="pt").to(device)

# Labels = Inputs (Standard Causal Language Modeling objective)
labels = tokens.input_ids.clone()

# Forward Pass
outputs = model(input_ids=tokens.input_ids, labels=labels)
loss = outputs.loss

print(f"Initial Loss: {loss.item():.4f}")

# Backward Pass
loss.backward()

# Check Gradients
print("\n--- Gradient Check ---")
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"Gradient found for: {name}")
        break # Just showing the first one to prove it works
else:
    print("No gradients found? Something is wrong.")

### Conclusion
You have successfully loaded a billion-parameter model into limited memory using **4-bit NF4 Quantization** and set it up for training using **LoRA**.

This architecture is exactly what powers tools like **Ollama**, **Llama.cpp**, and custom fine-tuned chatbots running on consumer hardware today.